In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

####Cheack the path of file
- dbutils.fs.ls('/FileStore/tables')
####Deleat a file or directory
- dbutils.fs.rm("dbfs:/FileStore/tables/BigMart_Sales-1.csv", recurse=True)


In [0]:
from pyspark.sql import SparkSession 

#create session
spark = SparkSession.builder.appName("BigMart_Sales_tutoria").getOrCreate()

In [0]:
spark

### Reading JSON file

In [0]:
df_json= spark.read.format("JSON").option('heaer',True)\
            .option('inferSchema',True)\
            .option('multiLine',False)\
            .load('/FileStore/tables/drivers.json')

In [0]:
df_json.display()

###Reading CSV file

In [0]:
df = spark.read.format("csv").option('header', True).option('inferSchema', True).load('/FileStore/tables/BigMart_Sales.csv')

In [0]:
df.display()

### DDL SCHEMA

In [0]:
df.printSchema()

In [0]:
my_ddl_schema= '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING,
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                '''
                #changing Item_Weight from Double to String
                #DDl Schema can only change data type 
                #Spark does NOT enforce NOT NULL in a DDL schema string.
                #💡 DDL string schema only defines data types, NOT constraints.

In [0]:
ddl_schema_df=spark.read.format('csv').option('header', True).schema(my_ddl_schema)\
                    .load('/FileStore/tables/BigMart_Sales.csv')

In [0]:
ddl_schema_df.printSchema()

In [0]:
ddl_schema_df.display()

### StructType() Schema

In [0]:
my_struct_type_schema=StructType([StructField('Item_Identifier',StringType(), False),
                                  StructField('Item_Weight',DoubleType(),False),
                                  StructField('Item_Fat_Content',StringType(),False),
                                  StructField('Item_Visibility',DoubleType(),False),
                                  StructField('Item_Type',StringType(),False),
                                  StructField('Item_MRP',DoubleType(),False),
                                  StructField('Outlet_Identifier',StringType(),False),
                                  StructField('Outlet_Establishment_Year',IntegerType(),False),
                                  StructField('Outlet_Size',StringType(),False),
                                  StructField('Outlet_Location_Type',StringType(),False),
                                  StructField('Outlet_Type',StringType(),False),
                                  StructField('Item_Outlet_Sales',DoubleType(),False),
                                  ])
#NotNull not working
#❌ Why nullable=False is Not Working?
#The nullable=False setting only affects the schema definition.
#Spark does NOT enforce NOT NULL constraints while reading data.
#If your dataset has NULL values, Spark will still load them without errors.


In [0]:
structType_df=spark.read.format('csv').option('header',True).option("inferSchema", False)\
                            .schema(my_struct_type_schema)\
                            .load('/FileStore/tables/BigMart_Sales.csv')

In [0]:
structType_df.printSchema()

In [0]:
structType_df.display()

### TRANSFORMATIONS

####SELECT

#####WayOne

In [0]:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

#####2nd Way

In [0]:
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

####ALIAS

In [0]:
df.select(col('Item_Identifier').alias ('Item_ID'),col('Item_Weight'),col('Item_Fat_Content').alias ('Fat_content')).display()

###Filter

####Scenario - 1

In [0]:
df.filter(col('Outlet_Location_Type')=='Tier 3').display()

In [0]:
df.filter(col('Outlet_Size').isNotNull()).display()

####Scenario - 2

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight')<10)).display()  

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type')=='Tier 1')).display()

####Scenario - 3

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()

###withColumnRenamed


In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

###withColumn

####Scenario - 1 new column

In [0]:
df= df.withColumn('flag',lit('new'))
df.display()

In [0]:
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

In [0]:
df.display()

####Scenario - Modify column

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","LF")).display()

In [0]:
df.display()

### Type Casting

In [0]:
df=df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

### Sort

In [0]:
df.sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(col('Item_Weight').asc()).display()

In [0]:
df.filter(col('Item_Weight').isNotNull()).sort(col('Item_Weight').asc()).display()

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending = [0,0]).display()

In [0]:
df.sort(['Item_weight'], ascending = [0]).display()

In [0]:
df.sort(['Item_Weight','Item_Visibility'],assending=[0,1]).display()

### Limit

In [0]:
df.limit(10).display() 

### DROP

In [0]:
df.drop('Item_Visibility').display()

In [0]:
df.drop('Item_Visibility','Item_Type').display()

### DRop_Duplicates

In [0]:
df.dropDuplicates().display()

In [0]:
df.drop_duplicates(subset=['Item_Type',"Item_Weight"]).display()

####distinct

In [0]:
df.distinct().display()

In [0]:
df.dropDuplicates(["Item_Type","Item_Weight"]).display()

### UNION and UNION BY NAME

####Preaparing  Dataframes

In [0]:
data1=[(1, 'kad'),
     (2,'sid')]
schema1 = 'ID STRING, Name STRING'

df1 = spark.createDataFrame(data1,schema1)

data2=[(3,'Rahul'),
     (4,'jas')]
schema2 = 'ID STRING, Name STRING'

df2 = spark.createDataFrame(data2,schema2)

df1.display()
df2.display()

####Union

In [0]:
df1.union(df2).display()

In [0]:
data1=[('kad',1),
       ('sid',2)]
schema1='name String, id String'
df1= spark.createDataFrame(data1, schema1)
df1.display()

In [0]:
df2.union(df1).display()

####Union By Name

In [0]:
df1.unionByName(df2).display()

In [0]:
df1.unionByName(df2).sort(col('id').desc()).display()

### Srting Function

####initcap,upper, lower


In [0]:
df.select(initcap('item_type')).display()

In [0]:
df.select(upper('Item_type')).display()

In [0]:
df.select(lower('item_type').alias('Lower_Item_Type')).display()

### Date Functions

####Current_Date

In [0]:
df=df.withColumn('Curr_date',current_date())
df.display()

####Dtae_add()

In [0]:
df=df.withColumn('week_after',date_add('Curr_date',7))
df.display()

####Date_sub()

In [0]:
df.withColumn('week_before',date_sub('Curr_date',7)).display()

In [0]:
df=df.withColumn('week_before',date_add('Curr_date',-7))
df.display()

####Datediff()

In [0]:
df=df.withColumn('Date_diff',datediff('week_after','Curr_date'))
df.display()

####Date_Format()

In [0]:
df = df.withColumn('week_before',date_format('week_before','dd-MM-yyyy'))

df.display()

### Handling Nulls

####Dropping NUlls

In [0]:
df.dropna('all').display()
#Removes rows where all columns have NULL (missing) values
#Keeps rows where at least one column has a non-null value

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Item_Weight']).display()

#### Filling Nulls

In [0]:
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvailable',subset=['Item_Weight']).display()

### SPLIT and Indexing

####SPLIT

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type', ' ')).display()

####Indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

### Explode

In [0]:
df_exp=df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

####array_contains

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).select('Outlet_Type','Type1_flag').display()

### GroupBY


####Scenario - 1

In [0]:
df.groupby('Item_Type').agg(sum('Item_MRP').alias('Sum_total')).display()

####Scenario - 2

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()

####Scenario - 3

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(avg('Item_MRP')).sort(col('avg(Item_MRP)').asc()).select('Item_Type','Outlet_Size','avg(Item_MRP)').display()

In [0]:
df.groupBy('Outlet_Size','Item_Type').agg(avg('Item_MRP')).sort(col('avg(Item_MRP)').asc()).select('Item_Type','Outlet_Size','avg(Item_MRP)').display()

####Scenario - 4

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP'),avg('Item_MRP')).display()

### Collect_List

In [0]:
df_collect=df_exp.withColumn('Outlet_Type',explode('Outlet_Type'))

In [0]:
df_collect.display()

In [0]:
df_collect.groupBy('Item_Identifier','Item_Weight','Item_Fat_Content','Item_Visibility','Item_Type','Outlet_Size','Outlet_Location_Type').agg(collect_list('Outlet_Type')).display()

### PIVOT


In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

### When-Otherwise

In [0]:
df=df.withColumn('veg_flag',when(col('Item_Type')=='Meat','Non-veg').otherwise('Veg'))
df.display()

In [0]:
df.withColumn('veg_exp_flag',when((col('veg_flag')=='Veg') & (col('Item_MRP')<100), 'Veg_Inexpensive')\
                                .when((col('veg_flag')=='Veg') & (col('Item_MRP')>=100), 'Veg_Expensive')\
                                .otherwise('Non_Veg')).display()

In [0]:
df = df.withColumn(
    "veg_exp_flag",
    when(col("veg_flag") == "Veg", 
         when(col("Item_MRP") < 100, "Veg_Inexpensive")
         .otherwise("Veg_Expensive"))
    .otherwise("Non_Veg")
)
df.display()

#nested when

In [0]:
df.display()

###JOINS

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

####innerjoin

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'inner').display()

In [0]:
df1.join(df2, "dept_id", "inner").display()

####Left Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

####RightJoin

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

#### Anti join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()

In [0]:
df2.join(df1,df1['dept_id']==df2['dept_id'],'anti').display()

### WINDOW FUNCTIONS

####ROW_NUMBER()

In [0]:
df.withColumn('rowCol', row_number().over(Window.orderBy('Item_Identifier'))).display()

####RANK VS DENSE RANK

In [0]:
df.withColumn('Rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('dense_Rank', dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()
    

####Cumulative Sum

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()
     

In [0]:
df.withColumn('totalsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()
     

### USER DEFINED FUNCTIONS (UDF)

####STEP - 1

In [0]:
def my_function(x):
    return x*x

####STEP - 2

In [0]:
my_udf = udf(my_function)

In [0]:
df.withColumn('mynewcol',my_udf('Item_MRP')).display()

### DATA WRITING

####CSV

In [0]:
df.write.format('csv')\
    .save('/FileStore/tables/CSV/data.csv')

####APPEND

In [0]:
df.write.format('csv')\
  .mode('append')\
  .save('/FileStore/tables/CSV/data.csv')

####Overwrite

In [0]:
df.write.format('csv')\
    .mode('overwrite')\
        .save('/FileStore/tables/CSV/data.csv')

####Error


In [0]:
df.write.format('csv')\
  .mode('error')\
    .save('/FileStore/tables/CSV/data.csv')

####Ignore

In [0]:
df.write.format('csv')\
  .mode('ignore')\
    .save('/FileStore/tables/CSV/data.csv')

### PARQUET file formet

In [0]:
df.write.format('parquet')\
  .mode('overwrite')\
    .save('/FileStore/tables/CSV/data.parquet')

# what is delta file format

#### TABLE


In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.saveAsTable('my_table')

### SPARK SQL

####createTempView

In [0]:
df.createOrReplaceTempView('my_view')

In [0]:
%sql

select * from my_view where Item_Fat_Content = 'LF'
     

In [0]:
df_sql = spark.sql("select * from my_view where Item_Fat_Content = 'LF'")

In [0]:
df_sql.display()